In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

shots = pd.read_pickle("../data/shots_wc2018.pkl")

# Remove penalties and penalty shootout kicks: they're a different kind of shot
shots = shots[(shots.shot_type != "Penalty") & (shots.period != 5)].copy()
print(shots.shape)

(1638, 95)


In [2]:
shots["x"] = shots.location.apply(lambda loc: loc[0])
shots["y"] = shots.location.apply(lambda loc: loc[1])
shots[["x", "y"]].describe()

,x,y
count,1638.000000,1638.000000
mean,103.769231,40.164225
std,8.838149,10.477915
min,59.000000,13.000000
25%,97.000000,32.000000
50%,105.000000,40.000000
75%,111.000000,48.000000
max,119.000000,69.000000


In [3]:
# Goal center is at (120, 40); the posts are at y = 36 and y = 44
shots["distance"] = np.sqrt((120 - shots.x) ** 2 + (40 - shots.y) ** 2)

def shot_angle(x, y):
    # Angle (in radians) formed between the two goalposts as seen from the shot location
    left = np.array([120 - x, 36 - y])
    right = np.array([120 - x, 44 - y])
    cos_a = np.dot(left, right) / (np.linalg.norm(left) * np.linalg.norm(right))
    return np.arccos(np.clip(cos_a, -1, 1))

shots["angle"] = shots.apply(lambda r: shot_angle(r.x, r.y), axis=1)
shots[["distance", "angle"]].describe()

,distance,angle
count,1638.000000,1638.000000
mean,19.333370,0.440839
std,8.804421,0.297653
min,1.414214,0.042154
25%,12.041595,0.252554
50%,18.867962,0.330925
75%,25.961510,0.539882
max,61.522354,2.622447


In [4]:
shots["goal"] = (shots.shot_outcome == "Goal").astype(int)
shots["is_header"] = (shots.shot_body_part == "Head").astype(int)

shots["shot_type"].value_counts()

shot_type
Open Play    1556
Free Kick      82
Name: count, dtype: int64

In [5]:
features = shots[["match_id", "player", "team", "x", "y", "distance", "angle",
                  "is_header", "shot_type", "goal", "shot_statsbomb_xg"]]
features.to_pickle("../data/features_wc2018.pkl")
print(features.shape, features.goal.mean())

(1638, 11) 0.08241758241758242


In [6]:
features.groupby(pd.cut(features.distance, [0, 6, 12, 18, 25, 40])).goal.mean()

distance
(0, 6]      0.392857
(6, 12]     0.154070
(12, 18]    0.080645
(18, 25]    0.042079
(25, 40]    0.028953
Name: goal, dtype: float64

In [7]:
def in_triangle(p, a, b, c):
    # True if point p lies inside triangle abc
    def sign(p1, p2, p3):
        return (p1[0]-p3[0])*(p2[1]-p3[1]) - (p2[0]-p3[0])*(p1[1]-p3[1])
    d1, d2, d3 = sign(p, a, b), sign(p, b, c), sign(p, c, a)
    has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
    has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)
    return not (has_neg and has_pos)

def freeze_features(row):
    ff = row.shot_freeze_frame
    if not isinstance(ff, list):
        return pd.Series({"defenders_in_cone": np.nan, "gk_dist_to_goal": np.nan, "nearest_defender": np.nan})

    shooter = (row.x, row.y)
    post_a, post_b = (120, 36), (120, 44)

    defenders = [f["location"] for f in ff
                 if not f["teammate"] and f["position"]["name"] != "Goalkeeper"]
    keepers = [f["location"] for f in ff
               if not f["teammate"] and f["position"]["name"] == "Goalkeeper"]

    in_cone = sum(in_triangle(d, shooter, post_a, post_b) for d in defenders)
    gk_dist = np.sqrt((120 - keepers[0][0])**2 + (40 - keepers[0][1])**2) if keepers else np.nan
    nearest = min((np.hypot(d[0]-row.x, d[1]-row.y) for d in defenders), default=np.nan)

    return pd.Series({"defenders_in_cone": in_cone, "gk_dist_to_goal": gk_dist, "nearest_defender": nearest})

ff_feats = shots.apply(freeze_features, axis=1)
shots = shots.join(ff_feats)

print(shots[["defenders_in_cone", "gk_dist_to_goal", "nearest_defender"]].describe())
print("Missing freeze frames:", shots.defenders_in_cone.isna().sum())

       defenders_in_cone  gk_dist_to_goal  nearest_defender
count        1638.000000      1637.000000       1638.000000
mean            0.791819         2.771353          3.347745
std             0.946979         2.463448          2.350618
min             0.000000         0.000000          0.000000
25%             0.000000         1.414214          2.000000
50%             1.000000         2.236068          2.828427
75%             1.000000         3.162278          4.123106
max             6.000000        58.008620         21.931712
Missing freeze frames: 0


In [8]:
features_v2 = shots[["match_id", "player", "team", "x", "y", "distance", "angle",
                     "is_header", "shot_type", "goal", "shot_statsbomb_xg",
                     "defenders_in_cone", "gk_dist_to_goal", "nearest_defender"]]
features_v2.to_pickle("../data/features_v2_wc2018.pkl")
print(features_v2.shape)

(1638, 14)
